# Syn-Cora Homophily Slider Results

Interactive prototype for inspecting how test-accuracy dispersion changes across synthetic Cora homophily levels. The table colors fold-std values relative to the `RandomKFold` baseline: baseline is white, lower dispersion is green, and higher dispersion is lightly red.

In [1]:
from pathlib import Path
import re

import pandas as pd

try:
    from IPython.display import HTML, display, clear_output
except ImportError:
    class HTML(str):
        pass

    def display(*objects):
        for obj in objects:
            print(obj)

    def clear_output(wait=False):
        pass

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "src/logs/runs").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing src/logs/runs.")


PROJECT_ROOT = find_project_root()

# Set this to a concrete list when you want to compare/load specific files.
# If None, all RunMetrics CSVs containing syn-cora rows are loaded.
RUN_METRICS_FILES = None

BASELINE_STRATEGY = "RandomKFold"
MODEL_ORDER = ["GCN", "GAT", "SAGE", "MixHop", "GPRGNN", "H2GCN", "MLP"]

RUNS_DIR = PROJECT_ROOT / "src/logs/runs"
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/jonas/Uni/SoSe26/Project/stratification_for_GNNs


## Load And Prepare Results

In [2]:
SYN_CORA_RE = re.compile(r"^syn-cora-h(?P<homophily>\d(?:\.\d+)?)-r(?P<realization>\d+)$")

PROPERTY_ORDER = {
    "Degree": 0,
    "NeighHet": 1,
    "PageRank": 2,
    "EigCentrality": 3,
    "Clustering": 4,
}

PROPERTY_LABELS = {
    "Degree": "Degree",
    "NeighHet": "Neigh. Het.",
    "PageRank": "PageRank",
    "EigCentrality": "Eigenvector",
    "Clustering": "Clustering",
}

STATIC_STRATEGIES = {
    "RandomKFold": ("Random", (0, 0, 0)),
    "LabelStratifiedKFold": ("Label", (0, 1, 0)),
    "WDES_Degree": ("WDES\nDegree", (2, 0, 0)),
    "WDES_NeighHet": ("WDES\nNeigh. Het.", (2, 1, 0)),
    "WDES_PageRank": ("WDES\nPageRank", (2, 2, 0)),
    "WDES_EigCentrality": ("WDES\nEigenvector", (2, 3, 0)),
    "WDES_Clustering": ("WDES\nClustering", (2, 4, 0)),
}


def resolve_run_metric_files():
    if RUN_METRICS_FILES is not None:
        paths = [Path(path).expanduser() for path in RUN_METRICS_FILES]
        return [path if path.is_absolute() else PROJECT_ROOT / path for path in paths]

    candidates = sorted(RUNS_DIR.glob("*_RunMetrics_*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No RunMetrics CSVs found in {RUNS_DIR}")

    syn_cora_paths = []
    for path in candidates:
        try:
            dataset_column = pd.read_csv(path, usecols=["Dataset"])
        except ValueError:
            continue
        contains_syn_cora = dataset_column["Dataset"].astype(str).str.startswith("syn-cora").any()
        if contains_syn_cora:
            syn_cora_paths.append(path)

    if not syn_cora_paths:
        raise FileNotFoundError(f"No RunMetrics CSVs with syn-cora rows found in {RUNS_DIR}")
    return syn_cora_paths


def parse_syn_cora_dataset(dataset_name):
    match = SYN_CORA_RE.match(str(dataset_name))
    if match is None:
        return None, None
    return float(match.group("homophily")), int(match.group("realization"))


def parse_strategy(stratifier):
    if stratifier in STATIC_STRATEGIES:
        label, sort_key = STATIC_STRATEGIES[stratifier]
        return stratifier, label, sort_key, None

    if stratifier.startswith("StratifiedKFoldDynamic_"):
        parts = stratifier.split("_")
        property_name = parts[1] if len(parts) > 1 else ""
        bin_part = parts[2] if len(parts) > 2 and parts[2].startswith("b") else ""
        selected_bins = int(bin_part[1:]) if bin_part[1:].isdigit() else None
        property_label = PROPERTY_LABELS.get(property_name, property_name)
        strategy = f"StratifiedKFoldDynamic_{property_name}"
        label = f"SKF dyn.\n{property_label}"
        sort_key = (1, PROPERTY_ORDER.get(property_name, 99), 0)
        return strategy, label, sort_key, selected_bins

    return stratifier, stratifier, (9, 99, 0), None


def format_bins(values):
    values = sorted({int(value) for value in values if pd.notna(value)})
    if not values:
        return ""
    if len(values) <= 3:
        return "bins " + ", ".join(map(str, values))
    return f"bins {values[0]}-{values[-1]}"


paths = resolve_run_metric_files()
frames = []
for path in paths:
    frame = pd.read_csv(path)
    frame["SourceFile"] = path.name
    frames.append(frame)
raw = pd.concat(frames, ignore_index=True)
print("Loaded:")
for path in paths:
    print(f"  {path}")

required_columns = {"Dataset", "StratificationType", "Fold_Seed", "Fold", "Model", "Init_Seed", "Test_Accuracy"}
missing_columns = required_columns - set(raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

parsed_datasets = raw["Dataset"].apply(parse_syn_cora_dataset)
raw["Homophily"] = parsed_datasets.apply(lambda value: value[0])
raw["Realization"] = parsed_datasets.apply(lambda value: value[1])
df = raw.dropna(subset=["Homophily", "Realization"]).copy()
if df.empty:
    raise ValueError("No syn-cora rows found. Expected Dataset values like syn-cora-h0.70-r1.")

parsed_strategies = df["StratificationType"].apply(parse_strategy)
df["Strategy"] = parsed_strategies.apply(lambda value: value[0])
df["StrategyLabel"] = parsed_strategies.apply(lambda value: value[1])
df["StrategySort"] = parsed_strategies.apply(lambda value: value[2])
df["SelectedBins"] = parsed_strategies.apply(lambda value: value[3])

df["AccuracyPercent"] = pd.to_numeric(df["Test_Accuracy"], errors="raise")
if df["AccuracyPercent"].max() <= 1.0:
    df["AccuracyPercent"] *= 100.0

print(f"Rows after syn-cora filter: {len(df):,}")
available_h_values = [round(float(value), 2) for value in sorted(df['Homophily'].unique())]
print(f"Available h values: {available_h_values}")
print(f"Models: {sorted(df['Model'].unique())}")

Loaded:
  /Users/jonas/Uni/SoSe26/Project/stratification_for_GNNs/src/logs/runs/0623-1718_RunMetrics_SYN-CORA.csv
  /Users/jonas/Uni/SoSe26/Project/stratification_for_GNNs/src/logs/runs/0623-1948_RunMetrics_SYN-CORA.csv
  /Users/jonas/Uni/SoSe26/Project/stratification_for_GNNs/src/logs/runs/0623-2139_RunMetrics_SYN-CORA.csv
  /Users/jonas/Uni/SoSe26/Project/stratification_for_GNNs/src/logs/runs/0624-1124_RunMetrics_SYN-CORA.csv
Rows after syn-cora filter: 27,500
Available h values: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
Models: ['GAT', 'GCN', 'GPRGNN', 'H2GCN', 'MLP', 'MixHop', 'SAGE']


In [3]:
# First aggregate across the five folds for each fixed split seed and model init.
fold_seed_stats = (
    df.groupby([
        "Homophily",
        "Realization",
        "Dataset",
        "Model",
        "Strategy",
        "StrategyLabel",
        "StrategySort",
        "Fold_Seed",
        "Init_Seed",
    ], as_index=False)
    .agg(
        Mean=("AccuracyPercent", "mean"),
        Std=("AccuracyPercent", "std"),
        NumFolds=("AccuracyPercent", "size"),
        BinSummary=("SelectedBins", format_bins),
    )
)

# Then aggregate these fold-std estimates for each h x model x strategy.
stats = (
    fold_seed_stats.groupby([
        "Homophily",
        "Model",
        "Strategy",
        "StrategyLabel",
        "StrategySort",
    ], as_index=False)
    .agg(
        Mean=("Mean", "mean"),
        Std=("Std", "mean"),
        NumEstimates=("Std", "size"),
        MinFolds=("NumFolds", "min"),
        MaxFolds=("NumFolds", "max"),
        BinSummary=("BinSummary", lambda values: "; ".join(sorted({v for v in values if v}))),
    )
)

strategy_order = (
    stats[["Strategy", "StrategyLabel", "StrategySort"]]
    .drop_duplicates("Strategy")
    .sort_values("StrategySort")
)
STRATEGIES = strategy_order["Strategy"].tolist()
STRATEGY_LABELS = dict(zip(strategy_order["Strategy"], strategy_order["StrategyLabel"]))

model_order = [model for model in MODEL_ORDER if model in stats["Model"].unique()]
model_order += [model for model in sorted(stats["Model"].unique()) if model not in model_order]

display(stats.head())
print(f"Aggregated rows: {len(stats):,}")
print(f"Strategies: {STRATEGIES}")

,Homophily,Model,Strategy,StrategyLabel,StrategySort,Mean,Std,NumEstimates,MinFolds,MaxFolds,BinSummary
0,0.0,GAT,LabelStratifiedKFold,Label,"(0, 1, 0)",31.234899,3.404856,10,5,5,
1,0.0,GAT,RandomKFold,Random,"(0, 0, 0)",30.946309,2.938140,10,5,5,
2,0.0,GAT,StratifiedKFoldDynamic_Clustering,SKF dyn.\nClustering,"(1, 4, 0)",30.691275,2.591106,10,5,5,bins 300
3,0.0,GAT,StratifiedKFoldDynamic_Degree,SKF dyn.\nDegree,"(1, 0, 0)",31.241611,2.739200,10,5,5,bins 300
4,0.0,GAT,StratifiedKFoldDynamic_EigCentrality,SKF dyn.\nEigenvector,"(1, 3, 0)",30.818792,2.522742,10,5,5,bins 300


Aggregated rows: 550
Strategies: ['RandomKFold', 'LabelStratifiedKFold', 'StratifiedKFoldDynamic_Degree', 'StratifiedKFoldDynamic_NeighHet', 'StratifiedKFoldDynamic_PageRank', 'StratifiedKFoldDynamic_EigCentrality', 'StratifiedKFoldDynamic_Clustering', 'StratifiedKFoldCategorical_PropLabelCluster']


## Interactive Table

In [4]:
def blend_channel(start, end, amount):
    return round(start + (end - start) * amount)


def blend_hex(start_hex, end_hex, amount):
    amount = max(0.0, min(1.0, float(amount)))
    start = tuple(int(start_hex[i:i + 2], 16) for i in (1, 3, 5))
    end = tuple(int(end_hex[i:i + 2], 16) for i in (1, 3, 5))
    mixed = tuple(blend_channel(s, e, amount) for s, e in zip(start, end))
    return "#" + "".join(f"{value:02x}" for value in mixed)


def table_for_h(h_value):
    subset = stats[stats["Homophily"] == h_value].copy()
    if subset.empty:
        raise ValueError(f"No rows available for h={h_value}")

    present_models = [model for model in model_order if model in subset["Model"].unique()]
    present_strategies = [strategy for strategy in STRATEGIES if strategy in subset["Strategy"].unique()]

    display_rows = []
    std_rows = []
    mean_rows = []
    n_values = []

    for model in present_models:
        model_subset = subset[subset["Model"] == model]
        display_row = {}
        std_row = {}
        mean_row = {}

        for strategy in present_strategies:
            values = model_subset[model_subset["Strategy"] == strategy]
            column_label = STRATEGY_LABELS[strategy]

            if values.empty:
                display_row[column_label] = ""
                std_row[column_label] = float("nan")
                mean_row[column_label] = float("nan")
                continue

            row = values.iloc[0]
            n_values.append(int(row["NumEstimates"]))
            bins = f"\n{row['BinSummary']}" if row["BinSummary"] else ""
            display_row[column_label] = f"Mean {row['Mean']:.2f}\nFold Std {row['Std']:.2f}{bins}"
            std_row[column_label] = float(row["Std"])
            mean_row[column_label] = float(row["Mean"])

        display_rows.append(pd.Series(display_row, name=model))
        std_rows.append(pd.Series(std_row, name=model))
        mean_rows.append(pd.Series(mean_row, name=model))

    display_table = pd.DataFrame(display_rows).fillna("")
    std_table = pd.DataFrame(std_rows)
    mean_table = pd.DataFrame(mean_rows)
    n_text = "n/a" if not n_values else (f"n={n_values[0]}" if len(set(n_values)) == 1 else f"n={min(n_values)}-{max(n_values)}")

    return display_table, std_table, mean_table, n_text


def style_dispersion_table(display_table, std_table):
    styles = pd.DataFrame("", index=display_table.index, columns=display_table.columns)

    for model in display_table.index:
        row_stds = std_table.loc[model].dropna()
        if row_stds.empty:
            continue

        ranked_columns = row_stds.sort_values(kind="mergesort").index.tolist()
        highlight_colors = {
            ranked_columns[0]: ("#006d2c", "#ffffff", "700"),
        }
        if len(ranked_columns) > 1:
            highlight_colors[ranked_columns[1]] = ("#74c476", "#111111", "600")
        if len(ranked_columns) > 2:
            highlight_colors[ranked_columns[2]] = ("#c7e9c0", "#111111", "600")

        for column in display_table.columns:
            value = std_table.loc[model, column]
            if pd.isna(value):
                continue

            style = "white-space: pre-line; text-align: center; border-bottom: 1px solid #dddddd;"

            background, text_color, font_weight = highlight_colors.get(
                column,
                ("#ffffff", "#111111", "400"),
            )
            styles.loc[model, column] = (
                style
                + f" background-color: {background};"
                + f" color: {text_color};"
                + f" font-weight: {font_weight};"
            )

    return styles


def table_html(h_value):
    display_table, std_table, mean_table, n_text = table_for_h(h_value)
    styled = (
        display_table.style
        .apply(lambda _: style_dispersion_table(display_table, std_table), axis=None)
        .set_table_styles([
            {"selector": "th", "props": [("font-weight", "700"), ("text-align", "center"), ("border-bottom", "1.5px solid #333333")]},
            {"selector": "td", "props": [("min-width", "125px"), ("padding", "9px 12px")]},
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "18px"), ("font-weight", "700"), ("padding", "8px")]},
        ])
        .set_caption(f"Syn-Cora h={h_value:.2f} | {n_text} split/init estimates")
    )
    return styled.to_html() + """
    <div style='font-size: 12px; margin-top: 8px;'>
      Color legend: dark green = lowest fold std, medium green = second lowest, light green = third lowest within each model row.
    </div>
    """


def render_table(h_value):
    display(HTML(table_html(h_value)))

In [5]:
homophily_values = sorted(stats["Homophily"].dropna().unique())

if widgets is None:
    print("ipywidgets is not installed. Showing the first available homophily table instead.")
    render_table(homophily_values[0])
else:
    slider = widgets.SelectionSlider(
        options=[(f"{value:.2f}", value) for value in homophily_values],
        value=homophily_values[0],
        description="h",
        continuous_update=False,
        layout=widgets.Layout(width="720px"),
        style={"description_width": "40px"},
    )
    table_widget = widgets.HTML(
        value=table_html(homophily_values[0]),
        layout=widgets.Layout(width="100%"),
    )

    def update(change=None):
        table_widget.value = table_html(slider.value)

    slider.observe(update, names="value")
    display(widgets.VBox([slider, table_widget]))